## Import 

In [1]:
import os
import gc
import math
import pickle 
import warnings
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import interp
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
from tableone import TableOne

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 500)

### General parameters

In [2]:
path_data  = "PATH TO DATA/eicu/Mortality24H/Data/"
race_path  = "PATH TO DATA/eICU/Data/csvExtract/"

### Reading Data

In [3]:
df_ehr_first_half  = pd.read_parquet(path_data + 'all_24h_data_first_half.parquet' , engine='pyarrow')
df_ehr_second_half = pd.read_parquet(path_data + 'all_24h_data_second_half.parquet', engine='pyarrow')
df_ehr = pd.concat([df_ehr_first_half, df_ehr_second_half])

In [4]:
print(df_ehr.patientunitstayid.nunique())
print(df_ehr.shape)

198190
(4383308, 721)


In [5]:
del df_ehr_first_half
del df_ehr_second_half
gc.collect()

0

### Drop Columns

In [6]:
del_diff_columns = [col for col in list(df_ehr.columns) if '_diff' in col]
del_tslm_columns = [col for col in list(df_ehr.columns) if '_tslm' in col]

del_diff_columns.extend(del_tslm_columns)
df_ehr = df_ehr.drop(columns=del_diff_columns)

print(df_ehr.patientunitstayid.nunique())
print(df_ehr.shape)

### Take first hours of ICU of patients with more than 24 hour LoS

In [9]:
observation_window = 24 * 60
observation_length = 24

In [10]:
def short_long_icustays(df, observation_window):
    
    short_stays = df[df.unitdischargeoffset < observation_window].copy()
    short_stays = short_stays.groupby('patientunitstayid').head(observation_length).reset_index(drop=True)

    long_stays = df[df.unitdischargeoffset >= observation_window].copy()
    long_stays = long_stays.groupby('patientunitstayid').head(observation_length).reset_index(drop=True)
    
    return long_stays, short_stays

In [11]:
df_long, df_short = short_long_icustays(df_ehr, observation_window)

In [12]:
print(df_long.patientunitstayid.nunique())
print(df_long.shape)

print(df_short.patientunitstayid.nunique())
print(df_short.shape)

132837
(3188088, 387)
65353
(1052946, 387)


In [13]:
del df_ehr
del df_short
gc.collect()

7

### Spliting Train - Validation - Test

In [14]:
df_long.sort_values(by=['patientunitstayid', 'Bins'], inplace=True)
df_long = df_long.reset_index(drop=True)

In [15]:
df_long.head(3)

### Variables Selection

In [16]:
selected_columns = ['uniquepid', 'patientunitstayid', 'Heart Rate', 'SpO2', 'O2 Saturation', 'Respiratory Rate', 
                    'Temperature (C)', 'Non-Invasive BP Mean', 'Non-Invasive BP Systolic', 
                    'Non-Invasive BP Diastolic', 'Glucose', 'creatinine', 'Base Excess', 'BUN', 
                    'anion gap', 'Bicarbonate',  'lactate', 'Hgb', 'Hct', 'pH', 'direct bilirubin',
                    'paO2', 'paCO2', 'AST (SGOT)', 'ALT (SGPT)', 'WBC x 1000', 'RBC', 'potassium', 'sodium', 
                    'chloride', 'magnesium', 'phosphate', 'ST1', 'ST2', 'ST3', 
                    'FiO2', 'PEEP', 'Tidal Volume', 'Urine_IO', 
                    'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
                    'GCS Total', 'Sedation Goal',
                    'age', 'gender', 'ethnicity', 
                    'unitdischargeoffset', 'unitdischargestatus', 'hospitaldischargestatus']

In [17]:
for col in selected_columns:
    temp_col = col + '_ind'
    
    if temp_col in list(df_long.columns):
        df_long.loc[df_long[temp_col] == 0, col] = np.nan

In [19]:
df_long = df_long[selected_columns]
df_long.head(3)

In [20]:
df_ehr = df_long.drop(['uniquepid', 'patientunitstayid', 'age', 'gender', 'ethnicity', 'unitdischargeoffset', 'hospitaldischargestatus'], axis=1)
df_demog = df_long[['uniquepid', 'patientunitstayid', 'age', 'gender', 'ethnicity', 'unitdischargeoffset', 'unitdischargestatus']]

### Create TableOne

In [21]:
columns = [ 'Heart Rate', 'SpO2', 'O2 Saturation', 'Respiratory Rate', 
            'Temperature (C)', 'Non-Invasive BP Mean', 'Non-Invasive BP Systolic', 
            'Non-Invasive BP Diastolic', 'Glucose', 'creatinine', 'Base Excess', 'BUN', 
            'anion gap', 'Bicarbonate',  'lactate', 'Hgb', 'Hct', 'pH', 'direct bilirubin',
            'paO2', 'paCO2', 'AST (SGOT)', 'ALT (SGPT)', 'WBC x 1000', 'RBC', 'potassium', 'sodium', 
            'chloride', 'magnesium', 'phosphate', 'ST1', 'ST2', 'ST3', 
            'FiO2', 'PEEP', 'Tidal Volume', 'Urine_IO', 
            'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
            'GCS Total', 'Sedation Goal',
            'unitdischargestatus']

In [22]:
categorical = ['Sedation Goal']

In [23]:
IQR = [ 'Heart Rate', 'SpO2', 'O2 Saturation', 'Respiratory Rate', 
        'Temperature (C)', 'Non-Invasive BP Mean', 'Non-Invasive BP Systolic', 
        'Non-Invasive BP Diastolic', 'Glucose', 'creatinine', 'Base Excess', 'BUN', 
        'anion gap', 'Bicarbonate',  'lactate', 'Hgb', 'Hct', 'pH', 'direct bilirubin',
        'paO2', 'paCO2', 'AST (SGOT)', 'ALT (SGPT)', 'WBC x 1000', 'RBC', 'potassium', 'sodium', 
        'chloride', 'magnesium', 'phosphate', 'ST1', 'ST2', 'ST3', 
        'FiO2', 'PEEP', 'Tidal Volume', 'Urine_IO', 
        'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
        'GCS Total']

In [24]:
EHR_table = TableOne(df_ehr, groupby='unitdischargestatus', columns=columns, categorical=categorical, pval=True, nonnormal=IQR)

In [25]:
EHR_table

Grouped by unitdischargestatus                                                                       
                                                                      Missing              Overall                    0                    1 P-Value
n                                                                                          3188088              3021912               166176        
Heart Rate, median [Q1,Q3]                                              88295     83.8 [72.0,97.2]     83.4 [71.8,96.6]    92.0 [77.3,107.2]  <0.001
SpO2, median [Q1,Q3]                                                   299413     97.3 [95.3,99.1]     97.3 [95.3,99.0]     97.4 [94.6,99.6]   0.017
O2 Saturation, median [Q1,Q3]                                         1159268     97.0 [95.0,99.0]     97.0 [95.0,99.0]     97.5 [95.0,99.7]  <0.001
Respiratory Rate, median [Q1,Q3]                                       258897     18.8 [16.0,22.5]     18.7 [15.9,22.3]     20.9 [17.0,25.8]  <0.001
Temperature (C), median [Q1,Q3]                                       2168496     36.9 [36.5,37.3]     36.9 [36.6,37.3]     36.7 [35.9,37.3]  <0.001
Non-Invasive BP Mean, median [Q1,Q3]                                   506906     79.2 [69.5,91.0]     79.8 [70.0,91.3]     74.0 [65.0,85.0]  <0.001
Non-Invasive BP Systolic, median [Q1,Q3]                               493010  118.0 [104.0,135.0]  118.5 [104.5,135.2]   109.5 [97.0,125.8]  <0.001
Non-Invasive BP Diastolic, median [Q1,Q3]                              493251     64.0 [55.2,74.0]     64.0 [55.5,74.3]     60.0 [52.0,70.0]  <0.001
Glucose, median [Q1,Q3]                                               2515882  135.0 [109.0,175.0]  135.0 [109.0,173.0]  143.0 [110.0,194.0]  <0.001
creatinine, median [Q1,Q3]                                            2984543        1.1 [0.8,1.8]        1.0 [0.8,1.7]        1.6 [1.0,2.6]  <0.001
Base Excess, median [Q1,Q3]                                           3098889      -1.4 [-5.0,2.0]      -1.0 [-4.7,2.3]      -4.7 [-9.9,0.6]  <0.001
BUN, median [Q1,Q3]                                                   2985942     20.0 [13.0,35.0]     20.0 [13.0,34.0]     30.0 [19.0,48.0]  <0.001
anion gap, median [Q1,Q3]                                             3026806      11.0 [8.0,14.0]      11.0 [8.0,14.0]     13.0 [10.0,17.0]  <0.001
Bicarbonate, median [Q1,Q3]                                           2903921     23.0 [20.0,26.0]     23.0 [20.1,26.0]     20.9 [17.0,24.5]  <0.001
lactate, median [Q1,Q3]                                               3117230        1.9 [1.2,3.3]        1.8 [1.1,2.9]        3.5 [1.9,6.9]  <0.001
Hgb, median [Q1,Q3]                                                   2971858      10.4 [8.8,12.2]      10.4 [8.8,12.2]      10.1 [8.6,12.0]  <0.001
Hct, median [Q1,Q3]                                                   2971858     31.5 [26.9,36.6]     31.6 [27.0,36.6]     30.9 [26.3,36.4]  <0.001
pH, median [Q1,Q3]                                                    3067823        7.4 [7.3,7.4]        7.4 [7.3,7.4]        7.3 [7.2,7.4]  <0.001
direct bilirubin, median [Q1,Q3]                                      3118511        0.2 [0.1,0.6]        0.2 [0.1,0.6]        0.4 [0.2,1.1]  <0.001
paO2, median [Q1,Q3]                                                  3065744   101.0 [77.0,149.0]   101.4 [77.0,148.0]    98.0 [72.0,155.0]  <0.001
paCO2, median [Q1,Q3]                                                 3067148     40.0 [34.4,47.5]     40.4 [35.0,47.5]     39.0 [32.0,47.4]  <0.001
AST (SGOT), median [Q1,Q3]                                            3119755     33.0 [20.0,72.0]     32.0 [19.0,66.0]    67.0 [34.0,166.0]  <0.001
ALT (SGPT), median [Q1,Q3]                                            3119859     27.0 [17.0,53.0]     27.0 [16.0,50.0]    42.0 [22.0,112.0]  <0.001
WBC x 1000, median [Q1,Q3]                                            3013614      11.0 [7.9,15.2]      10.8 [7.9,14.9]      13.3 [8.7,19.0]  <0.001
RBC, median [Q1,Q3]     

### Static Information

In [27]:
df_demog = df_demog.groupby('patientunitstayid').head(1)
df_demog.head()

### Read Race Dictionary

In [28]:
with open(race_path + 'ethnicity_dictionary.pkl', 'rb') as f:
    race_dictionary = pickle.load(f)

In [29]:
def replace_ethnicity_with_names(df, ethnicity_dict):
    
    inv_ethnicity_dict = {v: k for k, v in ethnicity_dict.items()}
    df['ethnicity'] = df['ethnicity'].map(inv_ethnicity_dict)
    
    return df

In [30]:
df_demog = replace_ethnicity_with_names(df_demog, race_dictionary)
df_demog.loc[df_demog.ethnicity == 'nodx', 'ethnicity'] = 'Other/Unknown'

In [31]:
df_demog['unitdischargeoffset'] = np.round((df_demog['unitdischargeoffset'] / 60), 1)

In [32]:
df_demog.head()

In [33]:
columns = ['age', 'gender', 'ethnicity', 'unitdischargeoffset', 'unitdischargestatus']

categorical = ['gender', 'ethnicity']

IQR = ['age', 'unitdischargeoffset']

In [34]:
demog_table = TableOne(df_demog, groupby='unitdischargestatus', columns=columns, categorical=categorical, pval=True, nonnormal=IQR)

In [35]:
demog_table

Grouped by unitdischargestatus                                                               
                                                                            Missing           Overall                 0                  1 P-Value
n                                                                                              132837            125913               6924        
age, median [Q1,Q3]                                                              21  66.0 [54.0,76.0]  65.0 [53.0,76.0]   69.0 [58.0,79.0]  <0.001
gender, n (%)                       0                                             0          56 (0.0)          53 (0.0)            3 (0.0)   0.459
                                    1                                                    72073 (54.3)      68266 (54.2)        3807 (55.0)        
                                    2                                                    60708 (45.7)      57594 (45.7)        3114 (45.0)        
ethnicity, n (%)                    African American                              0      14970 (11.3)      14213 (11.3)         757 (10.9)   0.074
                                    Asian                                                  2223 (1.7)        2091 (1.7)          132 (1.9)        
                                    Caucasian                                           102327 (77.0)      96977 (77.0)        5350 (77.3)        
                                    Hispanic                                               4938 (3.7)        4716 (3.7)          222 (3.2)        
                                    Native American                                         873 (0.7)         823 (0.7)           50 (0.7)        
                                    Other/Unknown                                          7506 (5.7)        7093 (5.6)          413 (6.0)        
unitdischargeoffset, median [Q1,Q3]                                               0  55.6 [38.2,97.3]  54.4 [37.9,95.2]  85.0 [46.1,169.4]  <0.001
[1] Chi-squared tests for the following variables may be invalid due to the low number of observations: gender.

### Save Tables

In [36]:
# EHR_table.to_csv('./TableONe_EHR_ICU_Mortality.csv')
# demog_table.to_csv('./TableONe_DEMOG_ICU_Mortality.csv')